In [1]:
import os
import torch
from torch.utils.data import Dataset, DataLoader

import models
import data_loader
import base
import meta
import plot
import evaluation
from trainers import ANN, LSTM, MetaModel

In [2]:
TICKER      = "AAPL"      # Change to any ticker you want
START_DATE  = "2015-01-01"
END_DATE    = "2024-01-01"
WINDOW_SIZE = 30          # How many past days the model sees
BATCH_SIZE  = 64
N_SPLITS    = 5           # TimeSeriesSplit folds; we use the last one
EPOCHS = 200


OUTPUT_DIR  = "outputs"
EVALUATION_DIR = "evaluation"
os.makedirs(OUTPUT_DIR, exist_ok=True)
os.makedirs(EVALUATION_DIR, exist_ok=True)

### Data Loader

In [3]:
# Downloading and building features from data
df = data_loader.download_data(TICKER, START_DATE, END_DATE)
data = data_loader.build_features(df)
n_features = data.shape[1]
scaled, f_scaler, c_scaler = data_loader.normalize(data)

# Creatong sequences and splitting dataset for training
X, y = data_loader.create_sequences(scaled, WINDOW_SIZE)
X_train, X_test, y_train, y_test = data_loader.split_data(X, y)

# Splitting into train and test datasets
train_loader = DataLoader(
    models.StockDataset(X_train, y_train), 
    batch_size=BATCH_SIZE, shuffle=True
    )
test_loader  = DataLoader(
    models.StockDataset(X_test,  y_test), 
    batch_size=BATCH_SIZE, shuffle=False
    )

[*********************100%***********************]  1 of 1 completed

Downloaded 2264 rows for AAPL
Train: 1862 samples | Test: 372 samples


### Models Loader

In [4]:
lstm_model = LSTM(n_features=n_features)
ann_model  = ANN(n_features=n_features, seq_len=WINDOW_SIZE)
meta_model = MetaModel()

params = (sum(p.numel() for p in lstm_model.parameters()) +
                sum(p.numel() for p in ann_model.parameters()) +
                sum(p.numel() for p in meta_model.parameters()))

In [5]:
params

150705

### Training MetaModel Process

In [6]:
# LSTM and ANN model training
lstm_model, lstm_tr_loss, lstm_val_loss = base.train(
    lstm_model, train_loader, test_loader, name="LSTM", epochs=EPOCHS
)
ann_model, ann_tr_loss, ann_val_loss = base.train(
    ann_model, train_loader, test_loader, name="ANN", epochs=EPOCHS
)

# Get predictions from each model
lstm_tr_preds, ann_tr_preds, y_tr_meta = base.predict(
    lstm_model, ann_model, train_loader
)
lstm_val_preds, ann_val_preds, y_val_meta = base.predict(
    lstm_model, ann_model, test_loader
)


Training LSTM on cpu
  Epoch  10/200 | Train MSE: 0.000703 | Val MSE: 0.000850
  Epoch  20/200 | Train MSE: 0.000644 | Val MSE: 0.003046
  Epoch  30/200 | Train MSE: 0.000483 | Val MSE: 0.000937
  Epoch  40/200 | Train MSE: 0.000576 | Val MSE: 0.000959
  Epoch  50/200 | Train MSE: 0.000471 | Val MSE: 0.000530
  Epoch  60/200 | Train MSE: 0.000480 | Val MSE: 0.001085
  Epoch  70/200 | Train MSE: 0.000434 | Val MSE: 0.000766
  Epoch  80/200 | Train MSE: 0.000458 | Val MSE: 0.000587
  Epoch  90/200 | Train MSE: 0.000430 | Val MSE: 0.000609
  Epoch 100/200 | Train MSE: 0.000449 | Val MSE: 0.000611
  Epoch 110/200 | Train MSE: 0.000432 | Val MSE: 0.000647
  Epoch 120/200 | Train MSE: 0.000446 | Val MSE: 0.000620
  Epoch 130/200 | Train MSE: 0.000444 | Val MSE: 0.000621
  Epoch 140/200 | Train MSE: 0.000428 | Val MSE: 0.000618
  Epoch 150/200 | Train MSE: 0.000464 | Val MSE: 0.000619
  Epoch 160/200 | Train MSE: 0.000452 | Val MSE: 0.000621
  Epoch 170/200 | Train MSE: 0.000424 | Val MSE: 0

In [7]:
# Using previous LSTM and ANN predicted results to train the meta-model
meta_model, meta_tr_loss, meta_val_loss = meta.train(
        meta_model,
        lstm_tr_preds, ann_tr_preds, y_tr_meta,
        lstm_val_preds, ann_val_preds, y_val_meta,
        epochs=EPOCHS
    )

Training Meta-Model (Stacking Layer)
  Epoch  10/200 | Train MSE: 0.093137 | Val MSE: 0.368553
  Epoch  20/200 | Train MSE: 0.037878 | Val MSE: 0.151881
  Epoch  30/200 | Train MSE: 0.012551 | Val MSE: 0.051033
  Epoch  40/200 | Train MSE: 0.003320 | Val MSE: 0.013836
  Epoch  50/200 | Train MSE: 0.000757 | Val MSE: 0.003422
  Epoch  60/200 | Train MSE: 0.000246 | Val MSE: 0.001110
  Epoch  70/200 | Train MSE: 0.000178 | Val MSE: 0.000705
  Epoch  80/200 | Train MSE: 0.000172 | Val MSE: 0.000619
  Epoch  90/200 | Train MSE: 0.000172 | Val MSE: 0.000620
  Epoch 100/200 | Train MSE: 0.000172 | Val MSE: 0.000609
  Epoch 110/200 | Train MSE: 0.000172 | Val MSE: 0.000612
  Epoch 120/200 | Train MSE: 0.000172 | Val MSE: 0.000615
  Epoch 130/200 | Train MSE: 0.000172 | Val MSE: 0.000615
  Epoch 140/200 | Train MSE: 0.000172 | Val MSE: 0.000623
  Epoch 150/200 | Train MSE: 0.000172 | Val MSE: 0.000606
  Epoch 160/200 | Train MSE: 0.000172 | Val MSE: 0.000611
  Epoch 170/200 | Train MSE: 0.0001

### Evaluating all models

In [8]:
# Get scaled predictions
lstm_pred_s, y_true_s = evaluation.predict(lstm_model, test_loader)
ann_pred_s, _ = evaluation.predict(ann_model,  test_loader)
ens_pred_s, _ = evaluation.ensemble_predictions(
    lstm_model, ann_model, meta_model, test_loader
    )

In [9]:
# Inverse-transform to real $ values
y_true_r    = evaluation.inverse_scale(y_true_s,   c_scaler)
lstm_pred_r = evaluation.inverse_scale(lstm_pred_s, c_scaler)
ann_pred_r  = evaluation.inverse_scale(ann_pred_s,  c_scaler)
ens_pred_r  = evaluation.inverse_scale(ens_pred_s,  c_scaler)

In [10]:
# Compute and display metrics (real $ scale)
print("\n  ─── Test Set Performance (real $ scale) ───")
print(f"  {'Model':<25} {'R²':>8}  {'MAE':>8}  {'MSE':>10}  {'RMSE':>8}")
results = {}
results["LSTM"]     = evaluation.compute_metrics(y_true_r, lstm_pred_r, "LSTM")
results["ANN"]      = evaluation.compute_metrics(y_true_r, ann_pred_r,  "ANN")
results["LSTM+ANN"] = evaluation.compute_metrics(y_true_r, ens_pred_r,  "LSTM+ANN (Ensemble)")


  ─── Test Set Performance (real $ scale) ───
  Model                           R²       MAE         MSE      RMSE
  LSTM                      R²=+0.9593  MAE=3.0631  MSE=14.2522  RMSE=3.7752
  ANN                       R²=+0.9320  MAE=4.1259  MSE=23.7782  RMSE=4.8763
  LSTM+ANN (Ensemble)       R²=+0.9487  MAE=3.5336  MSE=17.9413  RMSE=4.2357


In [11]:
# Improvement summary
r2_ann = results["ANN"]["R2"]
r2_ens = results["LSTM+ANN"]["R2"]
if r2_ann > 0:
    improvement = (r2_ens - r2_ann) / abs(r2_ann) * 100
    print(f"\n  Ensemble R² improvement over ANN: {improvement:+.1f}%")


  Ensemble R² improvement over ANN: +1.8%


### Saving plots

In [12]:
plot.training_curves(
    {"LSTM": (lstm_tr_loss, lstm_val_loss),
        "ANN":  (ann_tr_loss,  ann_val_loss),
        "Meta": (meta_tr_loss, meta_val_loss)},
    save_path=f"{OUTPUT_DIR}/training_curves.png"
)
plot.predictions(
    y_true_r,
    {"LSTM": lstm_pred_r, "ANN": ann_pred_r, "LSTM+ANN": ens_pred_r},
    ticker=TICKER,
    save_path=f"{OUTPUT_DIR}/predictions.png"
)
plot.metrict_comparison(
    results,
    save_path=f"{OUTPUT_DIR}/metrics_comparison.png"
)

# Save model weights
torch.save(lstm_model.state_dict(), f"{OUTPUT_DIR}/lstm_weights.pt")
torch.save(ann_model.state_dict(),  f"{OUTPUT_DIR}/ann_weights.pt")
torch.save(meta_model.state_dict(), f"{OUTPUT_DIR}/meta_weights.pt")

print(f"  Weights saved to {OUTPUT_DIR}/")

print("\n" + "="*60)
print("  Training complete.")
print(f"  Best model: LSTM+ANN Ensemble")
print(f"  R² = {r2_ens:.4f} | RMSE = {results['LSTM+ANN']['RMSE']:.4f}")
print("="*60 + "\n")

  Saved: outputs/training_curves.png
  Saved: outputs/predictions.png
  Saved: outputs/metrics_comparison.png
  Weights saved to outputs/

  Training complete.
  Best model: LSTM+ANN Ensemble
  R² = 0.9487 | RMSE = 4.2357

